# Africa–Europe UN General Debate: NLP & Machine Learning

Professional reconstruction of the **Data Analytics & Artificial Intelligence (2023/24)** coursework. The original project compares African and European UN General Debate speeches using EDA, TF-IDF/cosine similarity, LDA, K-means, LSTM classification, sentiment analysis, and word clouds.


## 1. Setup

Install `requirements.txt` and run `python scripts/bootstrap_nltk.py`. The LSTM section is optional because TensorFlow is large; install `requirements-ai.txt` when reproducing it.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.preprocessing import load_ungd, add_text_features, filter_africa_europe
from src.comparative_analysis import (
    cross_region_similarity_summary,
    fit_joint_lda, lda_prevalence_by_group,
    kmeans_elbow, fit_kmeans_tfidf, top_terms_per_cluster,
    prepare_lstm_data, train_lstm_classifier,
)
from src.sentiment import add_sentiment_features
from src.visualization import (
    make_wordcloud, plot_cluster_distribution, plot_sentiment_over_time,
)


## 2. Data and regional sample


In [ ]:
df = load_ungd(ROOT / "data" / "un-general-debates.csv")
regional = add_text_features(filter_africa_europe(df))
regional["Continent"].value_counts()


The saved AI-course processed snapshot contains **3,826 speeches**: **2,159 Africa** and **1,667 Europe**. Country-to-continent choices are embedded in `src/preprocessing.py`; no external mapping spreadsheet is required.


## 3. Regional EDA and word clouds


In [ ]:
for continent in ["Africa", "Europe"]:
    part = regional.loc[regional["Continent"].eq(continent), "processed_text"]
    make_wordcloud(part, title=f"{continent}: high-frequency diplomatic vocabulary")


## 4. TF-IDF similarity — correcting the 0.2640 interpretation

The submitted notebook computed a full Europe×Africa cosine-similarity matrix and printed `cosine_sim[0][0]`. Therefore the famous **0.2640** value is **one speech-pair similarity**, not a corpus-level similarity.

The professional pipeline reports that value for provenance *and* two explicitly aggregate summaries.


In [ ]:
sim = cross_region_similarity_summary(
    regional["processed_text"].tolist(),
    regional["Continent"].tolist(),
    pair_sample_size=300,
    random_state=42,
)
pd.Series({
    "coursework_first_pair": sim.coursework_first_pair,
    "sampled_cross_region_pairwise_mean": sim.sampled_pairwise_mean,
    "regional_centroid_cosine": sim.centroid_cosine,
})


On the saved AI processed-data snapshot, the first-pair value reproduces approximately **0.2640**. Aggregate measures answer different questions and should not be conflated with it.


## 5. Joint LDA topic comparison


In [ ]:
tokens = [text.split() for text in regional["processed_text"]]
lda_model, lda_corpus, lda_dictionary = fit_joint_lda(
    tokens, num_topics=10, no_above=0.30, no_below=10,
    passes=50, random_state=0,
)

topic_words = pd.DataFrame({
    "topic": range(lda_model.num_topics),
    "top_words": [
        ", ".join(word for word, _ in lda_model.show_topic(i, topn=10))
        for i in range(lda_model.num_topics)
    ],
})
topic_prevalence = lda_prevalence_by_group(
    lda_model, lda_corpus, regional["Continent"].tolist()
)

topic_words, topic_prevalence


The source submission interprets high-prevalence topics around post-colonialism/racism and African conflicts; Kosovo/terrorism and Cold-War political language in Europe; health/food/education/MDGs; and governance/security/peacekeeping. Exact source topic words and submitted interpretations are recorded in `docs/findings.md`.


## 6. K-means clustering


In [ ]:
elbow = kmeans_elbow(
    regional["processed_text"].tolist(),
    k_values=range(1, 10),
    random_state=42,
)
elbow.plot(x="k", y="inertia", marker="o", legend=False, title="K-means elbow diagnostic")
plt.ylabel("Inertia")
plt.show()

km = fit_kmeans_tfidf(
    regional["processed_text"].tolist(),
    n_clusters=3,
    random_state=42,
)
regional = regional.copy()
regional["cluster"] = km.labels

cluster_counts = pd.crosstab(regional["Continent"], regional["cluster"])
cluster_terms = top_terms_per_cluster(km, top_n=10)
plot_cluster_distribution(km.labels)

cluster_counts, cluster_terms


The **saved original processed-data snapshot** contains cluster assignments of Africa `1194 / 51 / 914` and Europe `8 / 1140 / 519` across clusters 0/1/2. The narrative PDF reports slightly different counts from another run; the repository documents this discrepancy instead of silently selecting one.


## 7. LSTM continent classification (optional)

The submitted architecture used a 10,000-word vocabulary, sequence length 100, 100-dimensional embeddings, LSTM(128), Adam at 0.001, batch size 20, and 10 epochs. The submitted held-out accuracy was **85.1%** with confusion matrix `[[406, 46], [68, 246]]` for Europe/Africa.

The original notebook also reused the held-out test set as validation data during training. The professional helper below instead takes a validation slice from the training data and evaluates the test set only after fitting.


In [ ]:
RUN_LSTM = False

if RUN_LSTM:
    lstm_data = prepare_lstm_data(
        regional["processed_text"].tolist(),
        regional["Continent"].tolist(),
        vocab_size=10_000,
        sequence_length=100,
        test_size=0.20,
        random_state=42,
    )
    lstm_result = train_lstm_classifier(
        lstm_data,
        vocab_size=10_000,
        sequence_length=100,
        embedding_dim=100,
        lstm_units=128,
        learning_rate=0.001,
        batch_size=20,
        epochs=10,
        validation_split=0.15,
    )
    print("Professional rerun test accuracy:", lstm_result["test_accuracy"])
    print(lstm_result["confusion_matrix"])
else:
    print("LSTM skipped. Install requirements-ai.txt and set RUN_LSTM=True.")


The submitted training history reaches ~99–100% training accuracy while validation accuracy stays around 84–87%, indicating substantial **overfitting**. This is now stated explicitly in the portfolio rather than presenting 85.1% without qualification.


## 8. Corrected Africa–Europe sentiment comparison


In [ ]:
sentiment = add_sentiment_features(regional, text_column="text")
plot_sentiment_over_time(
    sentiment,
    group_column="Continent",
    title="Africa vs Europe: corrected sentence-level VADER sentiment",
)
pd.crosstab(sentiment["Continent"], sentiment["sentiment_label"], normalize="index")


The submitted report described more negative African sentiment in parts of the 1970s to mid-1980s and shared positive/negative vocabulary across continents. Because the coursework contains multiple sentiment implementations, these remain **source-reported findings**; future reproduction uses the corrected sentence-level pipeline above.


## 9. Reproducibility status

This notebook now contains executable code for **every major AI-course analysis**: regional construction, word clouds, similarity, LDA, K-means, optional full LSTM training/evaluation, and corrected sentiment. See `docs/qa.md` for what was source-reproduced, smoke-tested, and left optional due to heavy dependencies.
